# 05b · Experimentos extras no Mac (MPS): MedGemma 4B

**Experimentos extras** (run = `a2`, separados dos arquivos do artigo 1):
- **Artigo 2, prompt em português completo:** `main_pt` na amostra de avaliação inteira das duas bases.
- **Artigo 2, 2×2 de língua** (subamostra de robustez, 300 imagens por base): instrução em português com formato de
  resposta em inglês (`pt_instr_en_fmt`) e instrução em inglês com formato em português (`en_instr_pt_fmt`). As outras
  duas células do 2×2 são `main_en`/`r1` (já rodado) e `main_pt`/`a2` (acima). O prefill `ICDR: ` é o mesmo nas quatro.

Tempo estimado: ~2,5 h para o `main_pt` completo (≈1,9 s por imagem) e ~40 min para o 2×2.
Rode com o Mac na tomada e `caffeinate -dims` num Terminal. Grava a cada imagem e retoma de onde parou.

No fim, a última seção compara a checagem do piloto feita na nuvem (`piloto_cuda`) com a do Mac (`piloto_mps`),
depois que os resultados da nuvem forem copiados para `results/preds/`.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import config
pd.set_option("display.max_columns", 50, "display.width", 250, "display.max_rows", 200)
from vlmdr import cohort, infer, prompts
from vlmdr.runner import Runner

In [ ]:
MODELOS = ["medgemma_4b"]
ETAPAS = ("pt_completo", "dois_por_dois")
LIMITE = None          # ex.: 20 para um ensaio rápido; None = amostra inteira
print("dispositivo:", infer.device_name())
pr = prompts.load_prompts()
assert {"main_pt", "pt_instr_en_fmt", "en_instr_pt_fmt"} <= set(pr), "prompts do artigo 2 ausentes em settings/prompts.yaml"
cohorts = {ds: cohort.load(ds) for ds in ("brset", "mbrset")}
runner = Runner(cohorts, "local_extra", etapas=ETAPAS, limite=LIMITE)

In [ ]:
for m in MODELOS:
    runner.roda(m)
pd.DataFrame(runner.log)

## Conferência: `main_pt` no artigo 1 (`r1`, 300 imagens) × artigo 2 (`a2`)

Mesmas imagens, mesmo prompt: os graus devem ser idênticos (a execução é determinística).

In [ ]:
p = infer.load_preds("vlm_medgemma_4b_*_main_pt_*.csv")
w = p.pivot_table(index=["dataset", "image_id"], columns="run", values="grade_pred", aggfunc="first").dropna()
print("imagens nas duas execuções:", len(w), "| graus idênticos:", f"{(w['r1'] == w['a2']).mean():.1%}")

## Checagem MPS × CUDA (MedGemma 4B, piloto)

Precisa dos arquivos `vlm_medgemma_4b_*_main_en_piloto_cuda.csv` vindos da nuvem (notebook 05).

In [ ]:
rows = []
for ds in ("brset", "mbrset"):
    a, b = (config.PREDS / f"vlm_medgemma_4b_{ds}_main_en_piloto_{d}.csv" for d in ("mps", "cuda"))
    if not (a.exists() and b.exists()):
        print("faltando:", b.name if a.exists() else a.name)
        continue
    x = pd.read_csv(a, dtype={"image_id": str}).merge(pd.read_csv(b, dtype={"image_id": str}), on="image_id", suffixes=("_mps", "_cuda"))
    rows.append({"base": ds, "n": len(x), "grau idêntico %": 100 * (x.grade_pred_mps == x.grade_pred_cuda).mean(),
                 "edema idêntico %": 100 * (x.edema_pred_mps == x.edema_pred_cuda).mean(),
                 "max |Δ P(referenciável)|": (x.p_yes_mps - x.p_yes_cuda).abs().max()})
pd.DataFrame(rows)